In [ ]:
# Cell 1: GPU Verification & Imports
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("Using CPU")
    device = 'cpu'

# Import Cell-Path PINNs
from cell_path_pinns import CellPathModel, generate_synthetic_trajectory, generate_circular_trajectory

In [ ]:
# Cell 2: Generate Synthetic Microbe Trajectory
# Simulates a microbe moving towards a nutrient source with chemotactic behavior

t, x, y = generate_synthetic_trajectory(
    n_steps=200,
    dt=0.1,
    start_pos=(1, 1),
    target_pos=(8, 8),
    noise_scale=0.1,
    attraction_strength=0.05,
    seed=42
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(x, y, c=t, cmap='viridis', s=10)
plt.colorbar(label='Time')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Synthetic Microbe Trajectory')
plt.plot(x[0], y[0], 'go', markersize=10, label='Start')
plt.plot(x[-1], y[-1], 'r*', markersize=15, label='End')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t, x, 'b-', label='X', alpha=0.7)
plt.plot(t, y, 'r-', label='Y', alpha=0.7)
plt.xlabel('Time')
plt.ylabel('Position')
plt.title('Position vs Time')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Trajectory: {len(t)} points")
print(f"Time range: [{t.min():.2f}, {t.max():.2f}]")
print(f"X range: [{x.min():.2f}, {x.max():.2f}]")
print(f"Y range: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Cell 3: Train Cell-Path PINN Model
import time

# Initialize model
model = CellPathModel(
    hidden_dim=64,
    device=device,
    lambda_data=1.0,   # Data fitting weight
    lambda_geo=0.1,    # Geodesic (constant speed) weight
    lambda_chem=1.0    # Chemotactic weight
)

print(f"Training on device: {device}")
print("\nTraining Cell-Path PINN...")
print("="*50)

start_time = time.time()

model.fit(
    t, x, y,
    epochs=200,
    lr=1e-3,
    verbose=True,
    print_every=40
)

elapsed = time.time() - start_time

print("="*50)
print(f"✓ Training completed in {elapsed:.2f} seconds")

In [ ]:
# Cell 4: Visualize Training Progress
history = model.get_training_history()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Total loss
axes[0].semilogy(history['loss'], 'b-', linewidth=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training Loss (log scale)')
axes[0].grid(True, alpha=0.3)

# Individual losses
axes[1].semilogy(history['loss_data'], label='Data', linewidth=1)
axes[1].semilogy(history['loss_geo'], label='Geodesic', linewidth=1)
axes[1].semilogy(history['loss_chem'], label='Chemotactic', linewidth=1)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Components')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Final loss breakdown
final_losses = [
    history['loss_data'][-1],
    history['loss_geo'][-1],
    history['loss_chem'][-1]
]
labels = ['Data', 'Geodesic', 'Chemotactic']
axes[2].bar(labels, final_losses, color=['tab:blue', 'tab:orange', 'tab:green'])
axes[2].set_ylabel('Final Loss')
axes[2].set_title('Final Loss Breakdown')

plt.tight_layout()
plt.show()

print(f"Final total loss: {history['loss'][-1]:.6f}")
print(f"  - Data loss:    {history['loss_data'][-1]:.6f}")
print(f"  - Geodesic:     {history['loss_geo'][-1]:.6f}")
print(f"  - Chemotactic:  {history['loss_chem'][-1]:.6f}")

In [ ]:
# Cell 5: Predict and Compare
# Generate predictions at finer time resolution
t_pred = np.linspace(t.min(), t.max(), 300)
xy_pred = model.predict_paths(t_pred)
x_pred, y_pred = xy_pred[:, 0], xy_pred[:, 1]

plt.figure(figsize=(12, 5))

# Trajectory comparison
plt.subplot(1, 2, 1)
plt.scatter(x, y, c='black', s=20, alpha=0.5, label='True (observed)')
plt.plot(x_pred, y_pred, 'r-', linewidth=2, alpha=0.8, label='Predicted')
plt.plot(x[0], y[0], 'go', markersize=12, label='Start')
plt.plot(x[-1], y[-1], 'b*', markersize=15, label='End')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Trajectory: Observed vs Predicted')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')

# Time series comparison
plt.subplot(1, 2, 2)
plt.scatter(t, x, c='black', s=10, alpha=0.4, label='True X')
plt.scatter(t, y, c='gray', s=10, alpha=0.4, label='True Y')
plt.plot(t_pred, x_pred, 'r-', linewidth=2, label='Pred X')
plt.plot(t_pred, y_pred, 'b-', linewidth=2, label='Pred Y')
plt.xlabel('Time')
plt.ylabel('Position')
plt.title('Position vs Time')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute error
mse = model.compute_mse(t, x, y)
print(f"\nMean Squared Error: {mse:.6f}")

In [ ]:
# Cell 6: Visualize Learned Potential Field (Nutrient Landscape)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Show potential field at different times
times = [t.min(), t.mean(), t.max()]
titles = ['Early Time', 'Mid Time', 'Late Time']

for ax, time_val, title in zip(axes, times, titles):
    X, Y, U = model.get_potential_field(
        x_range=(x.min()-1, x.max()+1),
        y_range=(y.min()-1, y.max()+1),
        t=time_val,
        resolution=50
    )
    
    im = ax.contourf(X, Y, U, levels=20, cmap='viridis')
    ax.scatter(x, y, c='white', s=5, alpha=0.5)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(f'{title} (t={time_val:.1f})')
    plt.colorbar(im, ax=ax, label='U (potential)')

plt.suptitle('Learned Nutrient Potential Field U(x, y, t)', fontsize=14)
plt.tight_layout()
plt.show()

print("The potential field represents the learned nutrient concentration.")
print("Microbes should follow gradients of this field (chemotaxis).")

In [ ]:
# Cell 7: Test on Circular Trajectory (Geodesic Validation)
print("Testing on circular trajectory (perfect geodesic)...")
print("="*50)

# Generate circular trajectory
t_circ, x_circ, y_circ = generate_circular_trajectory(
    n_steps=100,
    radius=2.0,
    center=(0, 0),
    omega=1.0
)

# Train new model
model_circ = CellPathModel(hidden_dim=64, device=device)

start = time.time()
model_circ.fit(t_circ, x_circ, y_circ, epochs=200, lr=5e-3, verbose=True, print_every=50)
elapsed = time.time() - start

print(f"\n✓ Training completed in {elapsed:.2f} seconds")

# Predict
xy_pred_circ = model_circ.predict_paths(t_circ)

# Visualize
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.scatter(x_circ, y_circ, c='black', s=30, alpha=0.5, label='True')
plt.plot(xy_pred_circ[:,0], xy_pred_circ[:,1], 'r-', linewidth=2, label='Predicted')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Circular Trajectory')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Compute speed (should be constant for geodesic)
dt = t_circ[1] - t_circ[0]
vx = np.diff(xy_pred_circ[:,0]) / dt
vy = np.diff(xy_pred_circ[:,1]) / dt
speed = np.sqrt(vx**2 + vy**2)

plt.plot(t_circ[1:], speed, 'b-', linewidth=2)
plt.axhline(y=speed.mean(), color='r', linestyle='--', label=f'Mean: {speed.mean():.3f}')
plt.xlabel('Time')
plt.ylabel('Speed')
plt.title('Speed vs Time (should be constant)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

mse_circ = model_circ.compute_mse(t_circ, x_circ, y_circ)
print(f"\nCircular trajectory MSE: {mse_circ:.6f}")
print(f"Speed variation (std): {speed.std():.4f} (lower is better)")

In [ ]:
# Cell 8: Summary & Performance Statistics
print("="*60)
print("Cell-Path PINNs Demo Summary")
print("="*60)

print(f"\n📍 Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   Memory used: {torch.cuda.memory_allocated() / 1e6:.1f} MB")

print(f"\n📊 Performance:")
print(f"   Training time (200 epochs): {elapsed:.2f} seconds")
print(f"   Epochs per second: {200/elapsed:.1f}")

print(f"\n📈 Results:")
print(f"   Chemotactic trajectory MSE: {mse:.6f}")
print(f"   Circular trajectory MSE: {mse_circ:.6f}")

print(f"\n✅ Model successfully learned:")
print(f"   - Trajectory shape (data fitting)")
print(f"   - Approximately constant speed (geodesic constraint)")
print(f"   - Nutrient potential field (chemotaxis)")

print("\n" + "="*60)